# LangChain, LangGraph, and Langfuse

A practitioner brief on how model primitives, graph orchestration, and LLM observability fit together in production agent systems.

This notebook is **markdown only**: concepts, architecture, and reference snippets. It does not call a model.


## Scope

- How LangChain, LangGraph, and Langfuse divide work in a live agent stack
- LangChain primitives: chat models, prompts, LCEL pipes, tools, structured output, and `create_agent`
- LangGraph primitives: state, nodes, edges, reducers, persistence, interrupts, and hybrid graphs
- Langfuse primitives: traces, observations, sessions, prompt versions, scores, and experiments
- When to use a linear pipe, a tool-calling harness, or a named-node graph
- Current APIs versus deprecated paths (`LLMChain`, `create_react_agent`, `config_schema`)


## Overview

Production LLM systems fail in three different places: the **model call** is wrong, the **control flow** is wrong, or you **cannot see** what happened. These three products map onto those failure modes.

| Layer | Product | Job in production |
|---|---|---|
| Primitives | **LangChain** | Talk to models, format prompts, bind tools, parse structured output, run a standard tool loop |
| Orchestration | **LangGraph** | Named steps, shared state, branches, loops, retries, human gates, durable memory |
| Observability | **Langfuse** | Record every call, version prompts, score quality, compare experiments |

LangChain's own docs now draw the same split: LangChain is the agent framework (models, tools, `create_agent`); LangGraph is the low-level orchestration runtime (durable graphs, streaming, human-in-the-loop); tracing and evaluation sit in an observability product (LangSmith in the LangChain ecosystem, or Langfuse when you want an open-source, self-hostable platform).

`create_agent` is implemented **on top of LangGraph**. A standard tool loop does not require you to hand-draw nodes. A custom workflow — router, reviewer, map-reduce, mix of deterministic code and an agent — does.


## Intuition

**The situation.** A customer asks a question. Sometimes the answer is a single, well-shaped reply. Sometimes the system must look something up, call an internal API, wait for a person to approve a risky action, and only then respond. The same request can take one step or twelve, and two runs with the same wording will not always take the same path.

**The idea.** Think of a well-run kitchen during a dinner rush. The cook (the model) is skilled but only sees what you put on the ticket. Recipes and prep lists (reusable instructions) keep tickets consistent. The expediter (the workflow) decides whether a ticket is a simple salad, a multi-station entree, or something that needs the chef's sign-off. A kitchen camera and ticket printer (the recorder) capture what was ordered, what left the pass, how long each station took, and which tickets came back. You do not taste every plate to guess what went wrong last Tuesday — you look at the tickets.

**Why it matters.** Without a clear workflow, the cook improvises: extra steps, skipped checks, dishes that never leave the pass. Without a recorder, a bad reply is a ghost. You cannot tell whether the instructions were vague, the lookup returned the wrong page, a tool timed out, or a person never approved the action. Cost, delay, and silent errors pile up.

**What we are doing here**

- Ask the user, then decide if this is a straight reply, a lookup-and-act loop, or a multi-step workflow
- Give the model only the instructions and tools it needs for that path
- Pause for a human when the action is irreversible or regulated
- Save progress so a crash or an approval wait does not start from zero
- Record the full ticket so the next change is based on evidence, not memory

**What we are not doing.** This is not "add a chatbot and hope." A single model call is enough for many tickets. Graphs and recorders earn their keep when work has branches, tools, memory, or a quality bar you must prove.


## Architecture

![LangChain LangGraph Langfuse architecture](images/langchain_langgraph_langfuse_architecture.png)

**Flow**

1. **Prompt** — the app client sends the user request to the API gateway.
2. **Invoke** — the gateway starts the compiled graph at **Route Request**.
3. **Linear** — simple tickets go through an **LCEL Pipe** (prompt → model → parse).
4. **Tool loop** — tickets that need actions enter the **Agent Loop** (`create_agent` or an equivalent tool cycle).
5. **Call model** — both paths use **ChatOpenAI**; the agent may also hit the **Tool Registry**.
6. **Record** — the same run is written to a **Langfuse Trace** (and later scored in **Evaluation**).
7. **Persist** — graph state is saved by the **Checkpointer**; prompt text can be loaded from a **Prompt Store**.
8. **Answer** — after **Review Gate**, the reply returns through the gateway to the client.

**Legend:** User Interface (coral) · App plane (magenta) · Models (teal) · Data / Orchestration / Observability (magenta)


## Concepts

### How the three pieces fit

Treat them as **nested**, not competing:

1. **LangChain** supplies the ingredients: a chat model, a prompt template, tools, parsers.
2. **LangGraph** supplies the floor plan: which named step runs next, what shared state they read/write, when to stop, when to wait.
3. **Langfuse** supplies the flight recorder: one trace per user request, nested observations for model calls, tools, and retrieval, plus scores over time.

A linear FAQ answer may be only LangChain (an LCEL pipe) plus a Langfuse callback. A claims workflow with approval is a LangGraph `StateGraph` whose nodes call LCEL or `create_agent`, still with the same callback. You rarely want three separate "apps." You want one request path with three concerns.


### Glossary

| Term | Plain meaning | Lives in |
|---|---|---|
| Chat model | A model you talk to with messages (system / human / AI / tool), not a raw completion string | LangChain |
| Prompt template | Reusable instructions with `{variables}` filled at invoke time | LangChain, optionally Langfuse prompt manager |
| LCEL pipe | A left-to-right `|` pipeline: prompt → model → parser | LangChain |
| Tool | A typed function the model may call (search, ticket lookup, calculator) | LangChain |
| Agent harness | The loop around the model: prompt + tools + "call again until done" | LangChain `create_agent` |
| Middleware | Hooks that wrap that loop (retries, guardrails, logging, extra state) | LangChain agents |
| State | The shared clipboard the graph updates every step | LangGraph |
| Node | A function: read state, do work, return a **partial update** | LangGraph |
| Edge | The next-step rule: always go here, or choose based on state | LangGraph |
| Reducer | How two updates to the same field merge (replace vs append) | LangGraph |
| Checkpointer | Snapshots of state per **thread** so you can resume | LangGraph |
| Store | Long-term facts/preferences **across** threads | LangGraph |
| Interrupt | Pause the graph and wait for a human (or another system) | LangGraph |
| Trace | One user request, end to end | Langfuse |
| Observation | One nested step inside a trace (generation, tool, span, …) | Langfuse |
| Session | Several traces that belong to one conversation | Langfuse |
| Score | A quality label on a trace or observation | Langfuse |
| Dataset / experiment | A fixed set of cases you rerun when prompts or models change | Langfuse |


## LangChain

LangChain is the **component layer**. Current packaging:

| Package | Role |
|---|---|
| `langchain-core` | Interfaces, messages, prompts, runnables, LCEL |
| `langchain` | Agent harness (`create_agent`), tools helper, high-level APIs |
| `langchain-openai` | `ChatOpenAI` and other OpenAI integrations |
| `langgraph` | Graph runtime (pulled in because agents run on it) |

Install the integration you actually call (`langchain-openai` here). Do not import chat models from legacy `langchain.chat_models` factories. Docs also show `init_chat_model("openai:gpt-4o-mini")` for provider-portable strings; this repo's default remains in-cell `ChatOpenAI`.


### Chat models and the Responses API

`ChatOpenAI` is the chat-model class for OpenAI. Production-minded defaults for examples:

- Model: `gpt-4o-mini` (or `OPENAI_MODEL`)
- `temperature=0` unless you need variety
- `use_responses_api=True` so calls go to OpenAI's **Responses** API (the current generation surface for agentic apps: built-in tools, conversation state, reasoning items)

Auth is environment-based: `OPENAI_API_KEY` via `load_dotenv()`, never a hardcoded key.

The model is a **runnable**. You call `.invoke(...)`, `.stream(...)`, `.batch(...)`, or the async twins. You do not call `.run()` or `.predict()`.

**Structured output.** When the downstream system needs a schema (ticket fields, a route decision, a score), use `llm.with_structured_output(Schema)` rather than "please return JSON" plus a regex. That uses native structured output on the model.

**Tool binding.** `llm.bind_tools([my_tool])` advertises tools for a single model call. That is not yet an agent loop — the model may *request* a tool; your code or `create_agent` must *execute* it and feed the result back.


### Prompts

`ChatPromptTemplate.from_messages([...])` is the usual unit. Each tuple is a role plus a string that may contain `{variables}`:

- `"system"` — durable policy and role
- `"human"` — the current user text
- `"ai"` — prior model text (few-shot or history)
- `MessagesPlaceholder("history")` — a slot for a list of messages

Variable names in the template **must** match keys in `.invoke({...})`. A mismatch is a runtime error, not a silent empty string.

Prompts in code are fine for a first version. Once product and engineering both edit copy, move the text to Langfuse prompt management and keep the **shape** (variables, roles) in code.


### LCEL (LangChain Expression Language)

LCEL is the composition style for a **linear** (or simple DAG) pipeline. The pipe operator builds a runnable:

```text
prompt | model | parser
```

Each stage implements the same invoke/stream/batch/async contract. That is why streaming and retries come "for free" on the chain.

**Use LCEL when** the diagram is a straight line: format → generate → parse; few-shot; retrieve → generate (simple RAG); extract a schema.

**Do not use LCEL alone when** the model must call tools in a loop, when you need a named branch or cycle, or when you need checkpointed human approval. Those are `create_agent` or `StateGraph`.

Deprecated replacements: `LLMChain`, `SequentialChain`, `ConversationChain`, `.run()`, `.predict()`.


### Tools

A tool is a function plus a schema the model can read. The current helper is:

```python
from langchain.tools import tool

@tool
def get_order_status(order_id: str) -> str:
    """Return shipment status for an order id."""
    ...
```

Type hints **are** the input schema. The docstring is what the model uses to decide *when* to call it. Keep names in `snake_case`. Side effects (send email, charge a card) belong behind an explicit review step in the graph, not in a casually described tool.

Tools are inert until something executes tool calls: either your node code, or `create_agent`'s tools node.


### `create_agent` — the standard tool loop

LangChain v1's standard harness is:

```python
from langchain.agents import create_agent
```

**Agent = model + harness.** The harness is everything around the loop: system prompt, tools, middleware. The loop itself is:

1. Call the model with messages (and the system prompt).
2. If the AI message contains `tool_calls`, run those tools and append `ToolMessage`s.
3. Call the model again.
4. Stop when the model returns **no** tool calls (or a configured stop condition).

That graph is LangGraph under the hood, which is why checkpointing and interrupts work without you drawing nodes.

Pass `model=` as `ChatOpenAI(...)` in this repo (docs also accept `"openai:gpt-4o-mini"` strings). Pass `tools=` and `system_prompt=`. Invoke with a **messages** dict:

```python
agent.invoke({"messages": [{"role": "user", "content": "..."}]})
```

**Do not** use `from langgraph.prebuilt import create_react_agent`. That path is superseded by `create_agent`.

**Middleware** is how you customize the harness without forking it: retries, fallbacks, guardrails, extra logging, tool policies. Reach for a custom `StateGraph` only when middleware is not enough — routers across several agents, mixed deterministic ETL + LLM, map-reduce, or a review gate that is its own named node.


## LangGraph

LangGraph is the **orchestration runtime**. Official positioning: low-level, focused on durable execution, streaming, human-in-the-loop, and persistence. You can use it without LangChain; in this stack, nodes typically call LangChain runnables.

Core metaphor from the Graph API docs: **nodes do the work, edges decide what is next.** Execution is message-passing in super-steps (Pregel-style): a node runs, sends updates, downstream nodes become active. The graph halts when nothing is in flight.


### State, nodes, edges

**State** is a `TypedDict` (fast, usual default), a dataclass (defaults), or a Pydantic model (recursive validation, slower). Every node sees the current snapshot and returns a **dict of updates**, not a mutated object.

**Reducers** say how updates merge. With no reducer, last write wins (overwrite). Chat history almost always uses `add_messages` so each node *appends* rather than replacing the transcript. Shortcut: `MessagesState` already defines `messages` with that reducer.

**Nodes** are Python functions `(state) -> dict`. They may also receive `config` (`thread_id`, callbacks) and runtime context (store, stream writer). Keep side effects inside nodes, not in edge functions.

**Edges**

| Kind | API | When |
|---|---|---|
| Fixed | `add_edge(a, b)` | Always go `a → b` |
| Start / end | `START`, `END` | Entry and halt |
| Conditional | `add_conditional_edges(node, router, {"x": "node_x", "y": "node_y"})` | Branch on state |
| Fan-out | `Send` | Map-reduce: spawn many copies of a node |
| Combined hop + update | `Command` | Node returns both a state patch and a destination |

`Command` is the primitive for "update state **and** jump" from a node (or a tool). `Command(resume=...)` is how you continue after an interrupt. Cycles are allowed; they **must** have a path to `END` or you loop forever.

**Runtime context.** Immutable per-run data (`user_id`, db handle) belongs in `context_schema` on `StateGraph`, not the old `config_schema` name.


### Compile, invoke, stream

`StateGraph` is a **builder**. It cannot run until `.compile()`, which validates the graph (orphans, missing edges) and attaches checkpointers, stores, and interrupts.

```text
START → route → (linear_chain | agent_loop) → review → END
```

The compiled graph is a runnable:

- `.invoke(input, config={...})` — one result
- `.stream(...)` — tokens, node updates, or events, depending on stream mode

Chat-style graphs take `{"messages": [...]}`. Domain graphs take whatever keys you put on state (`topic`, `ticket_id`, …).


### Persistence: checkpointers vs stores

| | Checkpointer | Store |
|---|---|---|
| Saves | Full graph state snapshots | App-defined key/value items |
| Scope | One **thread** (`thread_id`) | Across threads (user, tenant, global) |
| Use | Conversation continuity, crash resume, time travel, HITL | Preferences, facts, playbooks |
| Dev | `InMemorySaver` | `InMemoryStore` |
| Prod | `PostgresSaver` (typical) | `PostgresStore` |

`InMemorySaver` / `MemorySaver` die on process restart. That is expected in a notebook; it is a defect in production.

You **must** pass `config={"configurable": {"thread_id": "..."}}` for a checkpointer to load the right snapshot. Keep `thread_id` reasonably short (Postgres columns cap around 255 characters).

Over long threads, checkpoints grow. Plan retention. Subgraphs have their own checkpoint namespace — parent graphs do not automatically see inner writes; use a store for data that must cross that boundary.


### Human-in-the-loop and memory

**Interrupts** pause before or after a node (or from inside a node via `interrupt()`). The checkpointer holds state. A reviewer inspects (and optionally edits) state, then you resume with `Command(resume=...)`. This is the right shape for "refund over $50," "send this email," or "deploy this change."

**Short-term memory** is the thread: messages + other state on the checkpointer.

**Long-term memory** is the store: write from a node when you learn a durable fact; read on later threads for the same user.

Streaming matters at this layer too: users should see node progress and tokens, not a spinner that hides a 40-second tool loop.


### Workflows vs agents vs hybrid

LangGraph docs distinguish:

- **Workflow** — you predetermined the path (prompt chain, router, reviewer). Nodes may still call an LLM.
- **Agent** — the model chooses tools and how many steps to take.

**Hybrid** is the production default for non-trivial apps: the **graph** owns routing, policy, and HITL; a **node** may run LCEL or `create_agent`.

| Shape | Composition |
|---|---|
| One prompt, one answer, maybe a parser | **LCEL** only — no graph |
| Model picks tools until done | **`create_agent`** |
| Named steps, branches, loops, mixed code + LLM | **LangGraph** `StateGraph` |
| Graph + a tool-using specialist at one step | **Hybrid** (`create_agent` or LCEL **inside** a node) |

Never flatten a branching workflow into a single pipe. Never wrap a one-step pipe in a graph "for consistency." Never reimplement the tool loop by hand when `create_agent` is enough.


## Langfuse

Langfuse is an **open-source LLM observability platform** (cloud or self-hosted). It is built on OpenTelemetry. The SDK batches in the background, so tracing should not sit on the user-facing latency path. Short-lived scripts **must** `flush()` before exit or the last traces never leave the process.

LangChain's productized counterpart is **LangSmith**. Choose Langfuse when you want MIT licensing, self-hosting, and a framework-agnostic data model. Choose LangSmith when you want the tightly managed LangChain/LangGraph ops surface. Many teams run LangGraph with Langfuse callbacks; that is a supported, first-class integration.


### Data model

| Concept | Meaning |
|---|---|
| **Observation** | One step: an LLM generation, a tool call, a span of work, a retrieval, … |
| **Trace** | All observations that share a `trace_id` — typically **one user request** |
| **Session** | Optional grouping of traces (a chat thread, a ticket) |

Trace-level attributes (`user_id`, `session_id`, `tags`, `metadata`, environment) are propagated onto observations so you can filter without joining tables.

**Observation types** (set automatically by the LangChain integration, or manually with `as_type=`):

| Type | Typical source |
|---|---|
| `generation` | Chat model call (tokens, cost, latency) |
| `span` | Generic unit of work |
| `event` | Instant marker |
| `agent` | Harness / graph deciding the flow |
| `tool` | `@tool` execution |
| `chain` | LCEL / intermediate glue |
| `retriever` | Vector or DB lookup |
| `embedding` | Embedding model call |
| `evaluator` | Quality scorer |
| `guardrail` | Safety / policy check |

A good LangGraph trace looks like the graph: **one span per node**, generations nested under the node that called the model, tools nested under the agent loop. A single blob named "LangGraph" with no children is under-instrumented.


### Tracing LangChain and LangGraph

The integration is **callbacks**, not a fork of your graph:

```python
from langfuse import get_client
from langfuse.langchain import CallbackHandler

langfuse = get_client()
handler = CallbackHandler()

result = graph.invoke(
    {"messages": [{"role": "user", "content": "..."}]},
    config={"callbacks": [handler]},
)
```

Same pattern for an LCEL chain or `create_agent`. For a server that should always trace, compile then `.with_config({"callbacks": [handler]})` so callers cannot forget.

Env (cloud example):

```text
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_BASE_URL=https://cloud.langfuse.com
```

Pin attributes per request via invoke metadata (`langfuse_user_id`, `langfuse_session_id`, `langfuse_tags`) or `propagate_attributes(...)` when you wrap LangChain inside `@observe()` / a parent span. That is how you join "this trace" to "this customer" and "this conversation."

For distributed systems, set a deterministic `trace_id` from the incoming request id so API logs and Langfuse share a key. `handler.last_trace_id` exists but is unsafe if one handler is reused across concurrent requests.


### Prompt management

Langfuse stores versioned prompts (text or chat) with labels such as `production` / `staging`. The SDK **caches** them in process, so a fetch is not a network hop on the hot path.

Why this exists: prompt copy and application deploys are owned by different people. A wording change should not wait on a release train. Keep **variables and roles** compatible with the code that fills them; a renamed `{variable}` is still a breaking change.

Link prompt versions to traces. Then an evaluation that says "helpfulness dropped last week" can be sliced by prompt version, not only by model.


### Evaluation

Evals turn traces into a loop you can run in CI and in production:

| Need | Langfuse feature |
|---|---|
| Humans rate live traces | Annotation queues, UI scores |
| Users thumbs-up / thumbs-down | User feedback scores |
| Repeatable regression set | **Datasets** (inputs + expected output) |
| Compare prompt/model/code | **Experiments** (UI or SDK) |
| Block a bad deploy | Experiments in CI |
| Deterministic checks | Code evaluators |
| Scalable quality labels | LLM-as-a-judge on traces or dataset runs |
| Trends | Score analytics and dashboards |

Typical production loop: sample traces → promote interesting failures into a dataset → change prompt or graph → run experiment → keep or revert. Online judges watch live traffic; offline datasets protect you before ship.

Cost, latency percentiles, and error rate belong on the same dashboards as quality. A cheaper model that doubles "not grounded" scores is not cheaper.


## When to use / when not to

| Situation | Prefer | Skip |
|---|---|---|
| FAQ, rewrite, classify, extract a schema | LCEL (+ `with_structured_output`) | Graph ceremony |
| "Search then answer" with no retries | LCEL with a retriever in the pipe | Agent loop |
| Model must call 1–N tools until done | `create_agent` | Hand-rolled while-loop |
| Router, reviewer, retries, mixed Python + LLM | LangGraph nodes and edges | Stuffing branches into LCEL |
| Approval, resume after crash, time travel | Graph + checkpointer + interrupt | Stateless invoke |
| Debug "what did the model see?" | Langfuse traces on day one | Logging only final strings |
| Product edits prompts weekly | Langfuse prompt manager | Prompts only in git |
| You cannot describe the steps | You do not have a design yet | A graph will not invent one |


## Comparison

| Concern | LCEL | `create_agent` | LangGraph `StateGraph` | Langfuse |
|---|---|---|---|---|
| Control flow | Linear / DAG | Tool cycle | Anything you draw | None (observes) |
| State | Invoke dict | Messages (+ middleware state) | Typed state + reducers | Trace attributes |
| Durability | None | Via underlying graph | Checkpointer / store | Historical traces |
| HITL | Awkward | Possible (graph features) | Native interrupts | Records the pause |
| Typical failure | Wrong parse | Tool loop never stops | Missing `END` / bad reducer | Missing callback / no flush |


## Output contract

For a production agent path, treat these as non-negotiable:

- **Invoke keys** match prompt variables and graph state keys.
- **Messages** stay a list with a reducer; do not overwrite history.
- **Tools** have types, a precise docstring, and no hidden side effects.
- **Structured steps** (routing, extraction) use `with_structured_output`, not ad-hoc JSON.
- **Every user request** has a trace id, a `user_id` / `session_id` when known, and environment tag `dev` | `staging` | `prod`.
- **Risky tools** sit behind a review node or interrupt.
- **Shutdown** flushes the Langfuse exporter.


## Reference implementation (snippets)

This brief is markdown-only. The snippets below are the current, recommended shapes — copy into an application; they are not executed here.

### Setup (every live cell in this repo's code notebooks)

```python
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, use_responses_api=True)
```


### LCEL pipe

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "Be concise. Answer only from the provided context if any."),
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()
print(chain.invoke({"question": "What is a checkpointer?"}))
```

Structured variant: `prompt | llm.with_structured_output(MySchema)`.


### Standard tool loop

```python
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Return a short weather string for a city."""
    return f"Sunny in {city}"

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a concise assistant. Use tools when they add facts.",
)
print(agent.invoke({
    "messages": [{"role": "user", "content": "Weather in Paris?"}],
}))
```


### LangGraph with LCEL inside a node (hybrid)

```python
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class State(TypedDict):
    topic: str
    outline: str
    draft: str

outline_chain = ChatPromptTemplate.from_messages([
    ("system", "Write a 3-bullet outline."),
    ("human", "{topic}"),
]) | llm | StrOutputParser()

def write_outline(state: State) -> dict:
    return {"outline": outline_chain.invoke({"topic": state["topic"]})}

graph = StateGraph(State)
graph.add_node("write_outline", write_outline)
graph.add_edge(START, "write_outline")
graph.add_edge("write_outline", END)
app = graph.compile()
print(app.invoke({"topic": "Checkpointers"}))
```

Chat graphs: subclass `MessagesState` (or declare `messages: Annotated[list, add_messages]`). Conditional routing: `add_conditional_edges("route", router_fn, {"linear": "lcel_pipe", "agent": "agent_loop"})`.


### Langfuse on invoke

```python
from langfuse import get_client
from langfuse.langchain import CallbackHandler

langfuse = get_client()
handler = CallbackHandler()

result = app.invoke(
    {"topic": "Checkpointers"},
    config={
        "callbacks": [handler],
        "metadata": {
            "langfuse_user_id": "user-123",
            "langfuse_session_id": "session-456",
            "langfuse_tags": ["support", "prod"],
        },
    },
)
langfuse.flush()
```


## Walkthrough

1. **Route** looks at the user text (rules or a tiny structured-output call) and returns `"linear"` or `"agent"`.
2. **LCEL pipe** handles deterministic tickets: one prompt, one model, one string or schema.
3. **Agent loop** handles tools. The model proposes calls; the harness executes them; messages accumulate until the model stops calling tools.
4. **Review gate** is a node or interrupt. Refunds, outbound messages, and writes to systems of record stop here.
5. **Checkpointer** stores the snapshot after each super-step so a crash or a human pause resumes the same thread.
6. **Langfuse** nests generations and tools under the node names. Evaluation later scores those traces or a dataset cloned from them.

If step 3 never ends, the bug is usually an unbounded tool policy or a tool that never answers the model's question — visible in the trace, not in the final string.


## Patterns

- Start with LCEL; promote to `create_agent` when you add tools; promote to a graph when you add named control flow.
- Put **policy in edges and interrupts**, not in a 2,000-word system prompt.
- One compiled graph per process; pass `thread_id` and callbacks **per request**.
- Name nodes after business steps (`check_eligibility`, not `node2`).
- Bind a small, high-quality tool set. Tool descriptions are prompts.
- Tag traces with feature, tenant, environment, and prompt version.
- Sample production traces into datasets monthly; that is your regression suite.
- Keep checkpointers transactional (Postgres). Keep prompt cache on by default.


## Pitfalls

| Wrong | Right |
|---|---|
| `LLMChain` / `.run()` / `.predict()` | LCEL `.invoke()` / `.stream()` |
| `from langgraph.prebuilt import create_react_agent` | `from langchain.agents import create_agent` |
| `StateGraph(..., config_schema=...)` | `context_schema=` |
| JSON instructions + regex parse as the extraction path | `with_structured_output` |
| Mutating `state["messages"]` in place | Return `{"messages": [new_message]}` and let the reducer merge |
| Forgetting `thread_id` with a checkpointer | Every invoke carries `configurable.thread_id` |
| `InMemorySaver` in production | `PostgresSaver` (or equivalent) |
| No Langfuse callback on the compiled graph | `callbacks=[handler]` or `.with_config(...)` |
| Process exit without `flush()` | Flush in workers, jobs, and notebooks |
| Reusing one `CallbackHandler` as global state for `last_trace_id` under concurrency | Parent span with an explicit `trace_id` |
| Graph for a single `prompt | llm | parser` | Stay on LCEL |
| LCEL for a tool-calling while-loop | `create_agent` |
| Hardcoded API keys | `.env` + `load_dotenv()` |


## Variants

- **Deep Agents** — a batteries-included harness on top of `create_agent` (planning, virtual filesystem, subagents). Use when you want that opinionated stack rather than assembling middleware yourself.
- **LangSmith** — drop-in tracing/eval/deploy for teams standardized on the LangChain SaaS. The graph code barely changes; the exporter does.
- **Functional API** — LangGraph also has a functional style. These briefs use the **Graph API** (`StateGraph`, named nodes) unless the topic is the Functional API itself.
- **Server-side tools** — OpenAI Responses built-in tools (web search, file search) bind on `ChatOpenAI` without a Python `@tool`. Still trace them; still gate irreversible actions.
- **Multi-agent** — supervisor or swarm as subgraphs. Each specialist can be `create_agent`; the parent graph owns handoff and shared state.


## Checklist

- [ ] Composition matches the diagram: pipe, harness, or named nodes — not all three glued at random
- [ ] `ChatOpenAI(..., use_responses_api=True)` (or a documented equivalent) and no hardcoded keys
- [ ] Prompt variables match invoke keys
- [ ] Tools: types, docstring, snake_case, no surprise writes
- [ ] Structured decisions use schemas, not prose-to-if-ladder
- [ ] Graph compiles; every cycle can reach `END`
- [ ] Reducers correct for lists vs scalars
- [ ] Production checkpointer + `thread_id`; in-memory only for local spikes
- [ ] Interrupts on irreversible tools
- [ ] Langfuse handler on every invoke; attributes for user/session/env
- [ ] `flush()` on short-lived processes
- [ ] Dataset + experiment before a prompt or model swap


## Worked example

A support product has three ticket shapes.

**Password reset copy** is a template fill. **Route** sends it to **LCEL Pipe**: system instructions + customer language → `ChatOpenAI` → string. No tools. Trace is a short chain with one generation. Cost should be tiny and stable.

**"Where is order 1842?"** needs a lookup. **Route** sends it to **Agent Loop** with `get_order_status`. The model calls the tool, reads the `ToolMessage`, and answers. The Langfuse trace should show `agent` → `generation` → `tool` → `generation`. If the tool 404s, you see it without asking the customer to screenshot the chat.

**"Refund $240 on order 1842"** is irreversible. The graph runs the same lookup, then **Review Gate** `interrupt()`s. The checkpointer holds `{order_id, amount, customer_id}`. An agent in the ops UI approves or edits amount. Resume writes to billing. Langfuse records the pause, the human score, and the final generation. A dataset of past refund traces becomes the regression pack when you change the refund prompt.

Same three products in all three tickets: LangChain for model/tools/pipe, LangGraph for route/loop/gate/memory, Langfuse for evidence.


## Takeaways

- LangChain is components and the standard tool harness; LangGraph is control flow and durability; Langfuse is evidence and eval.
- Pick composition from the shape of the work: **LCEL** for a line, **`create_agent`** for a tool cycle, **StateGraph** for named branches and gates, **hybrid** when a node *is* a pipe or an agent.
- `create_agent` already runs on LangGraph — do not recreate that loop, and do not use `create_react_agent`.
- State updates are patches; lists need reducers; production needs a real checkpointer and a `thread_id`.
- Tracing is a callback on invoke, not a rewrite. Flush, tag, and score, or the recorder is a toy.
- Observability without a dataset is a museum of traces. Datasets without tracing have no raw material.

**Sources (current docs):** [LangChain overview](https://docs.langchain.com/oss/python/langchain/overview) · [Agents / `create_agent`](https://docs.langchain.com/oss/python/langchain/agents) · [Install](https://docs.langchain.com/oss/python/langchain/install) · [Tools](https://docs.langchain.com/oss/python/langchain/tools) · [Structured output](https://docs.langchain.com/oss/python/langchain/structured-output) · [ChatOpenAI / Responses](https://docs.langchain.com/oss/python/integrations/chat/openai) · [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview) · [Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api) · [Use the graph API](https://docs.langchain.com/oss/python/langgraph/use-graph-api) · [Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents) · [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) · [Langfuse tracing](https://langfuse.com/docs/observability/overview) · [Data model](https://langfuse.com/docs/observability/data-model) · [Observation types](https://langfuse.com/docs/observability/features/observation-types) · [LangChain/LangGraph integration](https://langfuse.com/integrations/frameworks/langchain) · [Prompt management](https://langfuse.com/docs/prompt-management/overview) · [Evaluation](https://langfuse.com/docs/evaluation/overview)
